# Week 10 - Object Detection II: YOLO and Open-Vocabulary Detection

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Explain the **one-stage** detection idea and how YOLO differs from R-CNN.
- Run **YOLO** inference on images and video frames with the Ultralytics API.
- Describe how to **train** a detector on a custom dataset and how to evaluate it with **mAP**.
- Understand **open-vocabulary** detection with **YOLO-World** and text prompts.

### One-stage vs two-stage
Two-stage detectors first propose regions then classify them. **One-stage** detectors predict boxes and classes directly in a single network pass, trading a little accuracy for large speed gains - ideal for real-time robotics.

## 1. Setup
> Ultralytics downloads model weights on first use (~5-50 MB). GPU recommended.

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install ultralytics opencv-python matplotlib ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np
from cvhelpers import show, concept_map
from ultralytics import YOLO
print("Ultralytics ready")

## 2. How YOLO works (concept map)

In [ ]:
concept_map([
    "Resize image to a fixed size (e.g. 640x640)",
    "CNN backbone extracts features",
    "Neck (FPN/PAN) fuses multi-scale features",
    "Head predicts box, objectness and class per cell",
    "Decode predictions + non-maximum suppression",
    "Final boxes with labels and scores"
], title="YOLO one-stage detection pipeline")

## 3. Guided example - single-image inference
We use the small, fast **YOLO11n** model. The `results` object contains boxes, classes and confidences.

In [ ]:
model = YOLO("yolo11n.pt")   # auto-downloads the first time
img = cv2.imread("resources/images/test_image.jpeg")
results = model.predict(img, conf=0.25, verbose=False)[0]

print("Detections:", len(results.boxes))
for box in results.boxes:
    cls = model.names[int(box.cls)]
    print(f"{cls:15s} conf={float(box.conf):.2f}")

annotated = results.plot()
show(annotated, titles=["YOLO11n detections"], figsize=(8, 6))

###  Interactive exploration - confidence and IoU (NMS)
- **conf**: minimum score to keep a detection.
- **iou**: NMS overlap threshold; lower values remove more overlapping boxes.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def yolo_demo(conf=0.25, iou=0.5):
    r = model.predict(img, conf=conf, iou=iou, verbose=False)[0]
    show(r.plot(), titles=[f"conf={conf:.2f}  iou={iou:.2f}  ->  {len(r.boxes)} boxes"], figsize=(8, 6))

interact(yolo_demo,
         conf=widgets.FloatSlider(min=0.05, max=0.9, step=0.05, value=0.25),
         iou=widgets.FloatSlider(min=0.1, max=0.9, step=0.05, value=0.5))

## 4. Guided example - class filtering (traffic analysis)
YOLO is trained on COCO. We can restrict detection to vehicles for a counting application. COCO ids: car=2, motorcycle=3, bus=5, truck=7.

In [ ]:
vehicle_results = model.predict(img, classes=[2, 3, 5, 7], conf=0.2, verbose=False)[0]
names = [model.names[int(c)] for c in vehicle_results.boxes.cls]
print("Vehicles found:", names)
show(vehicle_results.plot(), titles=["Vehicles only"], figsize=(8, 6))

## 5. Guided example - video / frame inference
Video is processed frame by frame. In Colab we display a few sampled frames because interactive video windows (`cv2.imshow`) are not available.

In [ ]:
# Optional: upload a short clip as 'clip.mp4', then run this cell.
import os
if os.path.exists("clip.mp4"):
    cap = cv2.VideoCapture("clip.mp4")
    frames = []
    i = 0
    while cap.isOpened() and len(frames) < 3:
        ok, frame = cap.read()
        if not ok:
            break
        if i % 30 == 0:                       # sample every 30th frame
            r = model.predict(frame, conf=0.3, verbose=False)[0]
            frames.append(r.plot())
        i += 1
    cap.release()
    if frames:
        show(*frames, titles=[f"sampled frame {k}" for k in range(len(frames))], figsize=(15, 5))
else:
    print("No clip.mp4 found. Upload one, or use the vehicle video in resources/videos/ locally.")

## 6. Guided example - training on a custom dataset (template)
To detect your own objects (for example Malaysian banknotes), you need:

1. **Annotate** images (Roboflow, CVAT, Label Studio) in YOLO format: one `.txt` per image with `class cx cy w h` (normalised).
2. A **dataset YAML** pointing to `train`/`val` image folders and class names.
3. Train:
   ```python
   model = YOLO("yolo11n.pt")
   model.train(data="money/data.yaml", epochs=50, imgsz=640, batch=16)
   ```
4. **Evaluate** on the validation set and inspect the confusion matrix and precision-recall curves.

> The `projects/datasets/` folder is intentionally excluded from Git (large). Keep your datasets there locally.

In [ ]:
# Template - uncomment and adapt after adding your dataset
# model = YOLO('yolo11n.pt')
# results = model.train(data='projects/datasets/money/data.yaml', epochs=50, imgsz=640, batch=16)
# metrics = model.val()
# print('mAP50:', metrics.box.map50, '| mAP50-95:', metrics.box.map)
print('Training template ready. See markdown above.')

## 7. Guided example - open-vocabulary detection (YOLO-World)
Open-vocabulary models detect objects described by **free-form text**, without retraining. **YOLO-World** lets you `set_classes([...])`, which is powerful for flexible industrial inspection.

In [ ]:
# Optional - requires an extra model download.
try:
    from ultralytics import YOLOWorld
    wmodel = YOLOWorld("yolov8s-worldv2.pt")
    wmodel.set_classes(["banknote", "coin", "mobile phone"])
    wr = wmodel.predict(img, conf=0.25, verbose=False)[0]
    show(wr.plot(), titles=["YOLO-World (prompted)"], figsize=(8, 6))
except Exception as e:
    print("Open-vocabulary demo skipped:", e)

Other open-vocabulary detectors: **Grounding DINO** (paired with SAM for segmentation) and **OWL-ViT**. They accept text prompts such as *"the red valve"* and generalise far beyond a fixed label set.

## 8. Exercise (complete the code)

1. Run inference on `resources/images/money_counter.png`.
2. Count how many detections have confidence above **0.3**, **0.5** and **0.7**.
3. Plot a small bar chart of the counts. What does this tell you about the confidence threshold?

In [ ]:
# TODO: threshold analysis on money_counter.png


## 9. Challenge (independent)

Build a **real-time vehicle counter** concept: process a traffic video, track vehicles with YOLO + ByteTrack (`model.track(..., tracker='bytetrack.yaml')`), and count them crossing a horizontal line. You will revisit this in Week 13. For now, sketch the algorithm as a numbered list and identify where false counts could occur.

In [ ]:
# Your code here


## 10. Check your understanding (Q&A)

<details><summary><b>Q1. When would you choose YOLO over Faster R-CNN?</b></summary>

When inference speed and real-time operation matter more than the last few percent of accuracy - for example on drones, robots or embedded edge devices.
</details>

<details><summary><b>Q2. Why does changing the NMS IoU threshold change the number of boxes?</b></summary>

A lower IoU threshold removes boxes that overlap even moderately, merging more predictions; a higher threshold keeps more overlapping boxes.
</details>

<details><summary><b>Q3. What does 'open-vocabulary' mean?</b></summary>

The detector can be prompted with arbitrary text classes at inference time instead of being limited to a fixed, pre-trained label set.
</details>

<details><summary><b>Q4. What is the single biggest cause of poor custom-detector performance?</b></summary>

Poor or inconsistent annotation and insufficient/biased training data - not the model architecture.
</details>

## 11. Further reading & self-exploration
- Ultralytics documentation: https://docs.ultralytics.com/
- Ultralytics YOLO track mode: https://docs.ultralytics.com/modes/track/
- Ultralytics YOLO-World: https://docs.ultralytics.com/models/yolo-world/
- Roboflow annotation and datasets: https://roboflow.com/
- YOLO paper: Redmon et al. (2016), *You Only Look Once*.
- Grounding DINO: https://github.com/IDEA-Research/GroundingDINO
- Wikipedia - YOLO: https://en.wikipedia.org/wiki/You_Only_Look_Once

**Try next:** train YOLO on a small custom dataset (10 images per class is enough to learn the workflow).

## 12. Key takeaways
- YOLO = fast one-stage detection, ideal for real-time use.
- Tune **conf** (confidence) and **iou** (NMS) for your application.
- Custom detection depends on **annotation quality**.
- Evaluate with **mAP50 / mAP50-95**.
- **Open-vocabulary** detectors (YOLO-World, Grounding DINO) accept text prompts.